In [1]:
"""
Gray Level Co-occurrence Matrix (GLCM) para el Análisis de Texturas
Objetivo
Trabajar la Gray Level Co-occurrence Matrix (GLCM) como un descriptor estadístico clásico de texturas.
Importancia
En el cuaderno anterior, estudiamos los Patrones Binarios Locales (LBP), que codifican micropatrones locales alrededor de cada píxel. La GLCM ofrece una perspectiva diferente: en lugar de comparar un píxel central con sus vecinos, mide la frecuencia con la que aparecen pares de niveles de gris en un desplazamiento espacial determinado.
Ayuda a cuantificar la textura en términos de relaciones espaciales de niveles de gris.

"""

'\nGray Level Co-occurrence Matrix (GLCM) para el Análisis de Texturas\nObjetivo\nTrabajar la Gray Level Co-occurrence Matrix (GLCM) como un descriptor estadístico clásico de texturas.\nImportancia\nEn el cuaderno anterior, estudiamos los Patrones Binarios Locales (LBP), que codifican micropatrones locales alrededor de cada píxel. La GLCM ofrece una perspectiva diferente: en lugar de comparar un píxel central con sus vecinos, mide la frecuencia con la que aparecen pares de niveles de gris en un desplazamiento espacial determinado.\nAyuda a cuantificar la textura en términos de relaciones espaciales de niveles de gris.\n\n'

In [2]:
"""
Pasamos de patrones locales a relaciones de niveles de gris
La textura no solo se refiere a la variación de intensidad, sino también a cómo se distribuyen espacialmente los valores de intensidad.
La matriz de coocurrencia de niveles de gris (GLCM) captura esta idea al contar la frecuencia con la que dos niveles de gris aparecen juntos en una determinada:
- distancia
- orientación
Esto crea una matriz que resume las relaciones de intensidad espacial.
"""

'\nPasamos de patrones locales a relaciones de niveles de gris\nLa textura no solo se refiere a la variación de intensidad, sino también a cómo se distribuyen espacialmente los valores de intensidad.\nLa matriz de coocurrencia de niveles de gris (GLCM) captura esta idea al contar la frecuencia con la que dos niveles de gris aparecen juntos en una determinada:\n- distancia\n- orientación\nEsto crea una matriz que resume las relaciones de intensidad espacial.\n'

In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy.ndimage import uniform_filter
from skimage.feature import graycomatrix, graycoprops
from skimage.util import img_as_ubyte
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.decomposition import PCA

In [4]:
np.random.seed(42)
def show_images(images, titles, cmap='gray', figsize=(14, 8), save_path=None):
    n = len(images)
    cols = min(3, n)
    rows = int(np.ceil(n / cols))
    
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
    
    for ax in axes[len(images):]:
        ax.axis('off')
    
    plt.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()


def extract_random_patches(image, patch_size=32, n_patches=40):
    patches = []
    h, w = image.shape
    
    for _ in range(n_patches):
        i = np.random.randint(0, h - patch_size)
        j = np.random.randint(0, w - patch_size)
        patch = image[i:i+patch_size, j:j+patch_size]
        patches.append(patch)
    
    return patches


def quantize_image(image, levels=16):
    image_uint8 = img_as_ubyte(image)
    bins = np.linspace(0, 256, levels + 1)
    quantized = np.digitize(image_uint8, bins) - 1
    quantized[quantized == levels] = levels - 1
    return quantized.astype(np.uint8)


def compute_glcm(image, distances=[1], angles=[0], levels=16, symmetric=True, normed=True):
    image_q = quantize_image(image, levels=levels)
    glcm = graycomatrix(
        image_q,
        distances=distances,
        angles=angles,
        levels=levels,
        symmetric=symmetric,
        normed=normed
    )
    return glcm


def extract_glcm_features(image, distances=[1], angles=[0], levels=16):
    glcm = compute_glcm(image, distances=distances, angles=angles, levels=levels)
    
    features = {}
    props = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
    
    for prop in props:
        values = graycoprops(glcm, prop)
        features[prop] = float(values.mean())
    
    return features
